# Lab 02.3 — Ottimizzazione delle query

## Introduzione
Questo notebook mostra come Spark esegue le query in modo efficiente e quali leve si possono usare per migliorarne le prestazioni. I tempi misurati dipendono dalla macchina: contano i confronti relativi, non i valori assoluti.

### Obiettivi
1. Inizializzare la sessione e una funzione di cronometraggio.
2. Capire **shuffle** e numero di partizioni, e l'effetto di **Adaptive Query Execution (AQE)**.
3. Confrontare le strategie di **join** (broadcast hash vs sort-merge).
4. Distinguere `repartition` da `coalesce`.
5. Usare il **caching** dei DataFrame.
6. Chiudere la sessione.

## Step 1: Inizializzare la SparkSession e le utilità

Creiamo la sessione (stavolta senza ridurre `shuffle.partitions`: il default 200 serve a mostrare l'effetto di AQE), carichiamo le tre tabelle e definiamo `cronometra(etichetta, azione)`, una piccola funzione che esegue un'azione e ne stampa la durata.

In [ ]:
import os, time
from pyspark.sql import SparkSession, functions as F
import dati_shop

spark = (SparkSession.builder
         .appName("Lab02-Ottimizzazione")
         .master(os.environ.get("SPARK_MASTER", "local[*]"))
         .getOrCreate())
t = dati_shop.carica(spark)
ordini, prodotti, clienti = t["ordini"], t["prodotti"], t["clienti"]

def cronometra(etichetta, azione):
    t0 = time.time(); r = azione(); print(f"{etichetta:45s} {time.time() - t0:6.2f}s"); return r

## Step 2: Shuffle e numero di partizioni (AQE)

Dopo uno shuffle (`groupBy`, `join`, `orderBy`...) il numero di partizioni è `spark.sql.shuffle.partitions` (default **200**). Con pochi dati, 200 task minuscoli costano più del calcolo stesso.
**AQE** misura i dati reali dopo lo shuffle e **unisce** le partizioni troppo piccole.
Eseguiamo la stessa aggregazione con AQE spento e acceso, confrontando tempi e numero di partizioni del risultato.

In [ ]:
query = lambda: ordini.groupBy("cliente_id").agg(F.sum("quantita").alias("q"))

spark.conf.set("spark.sql.adaptive.enabled", "false")
df = query(); cronometra("AQE OFF", df.collect)
print("  partizioni risultato:", df.rdd.getNumPartitions())

spark.conf.set("spark.sql.adaptive.enabled", "true")
df = query(); cronometra("AQE ON", df.collect)
print("  partizioni risultato:", df.rdd.getNumPartitions(), "(unite da AQE: 'AQEShuffleRead coalesced' nel piano finale)")

### Punti chiave
* Il numero di partizioni dopo uno shuffle incide molto sui tempi.
* AQE regola a runtime il numero di partizioni in base ai dati reali.

## Step 3: Strategie di join

* **Broadcast Hash Join**: la tabella piccola viene copiata su ogni executor → nessuno shuffle della tabella grande.
* **Sort-Merge Join**: entrambe le tabelle vengono ripartizionate (shuffle) e ordinate per chiave → adatto a due tabelle grandi.

Spark sceglie in base a `spark.sql.autoBroadcastJoinThreshold` (10 MB); si può forzare con gli *hint*.
Disattiviamo AQE per confrontare le strategie "pure" (con AQE Spark potrebbe cambiarle a runtime), forziamo le due varianti con `hint("merge")` e `F.broadcast()`, cronometriamo e confrontiamo i piani con `explain()`; infine riattiviamo AQE.

In [ ]:
spark.conf.set("spark.sql.adaptive.enabled", "false")   # per confrontare le strategie "pure"

smj = ordini.join(prodotti.hint("merge"), "prodotto_id").groupBy("categoria").count()
bhj = ordini.join(F.broadcast(prodotti), "prodotto_id").groupBy("categoria").count()

cronometra("Sort-Merge Join", smj.collect)
cronometra("Broadcast Hash Join", bhj.collect)
smj.explain()
bhj.explain()
spark.conf.set("spark.sql.adaptive.enabled", "true")

### Punti chiave
* Il broadcast evita lo shuffle della tabella grande: ideale quando una tabella è piccola.
* Il sort-merge è la scelta generale per tabelle entrambe grandi.

## Step 4: `repartition` vs `coalesce`

* `repartition(n)` → shuffle completo, partizioni bilanciate (può aumentare o diminuire n).
* `coalesce(n)` → unisce partizioni esistenti **senza shuffle** (solo per diminuire n), utile prima di scrivere pochi file.

Stampiamo il numero di partizioni nei vari casi e il piano di `coalesce`.

In [ ]:
molte = ordini.repartition(16)
print("iniziali              :", ordini.rdd.getNumPartitions())
print("repartition(16)       :", molte.rdd.getNumPartitions())
print("repartition(16) → coalesce(4):", molte.coalesce(4).rdd.getNumPartitions())
molte.coalesce(4).explain()      # un solo 'Exchange' (quello del repartition): coalesce non fa shuffle

### Punti chiave
* Usa `repartition` per bilanciare o aumentare le partizioni, `coalesce` per ridurle in modo economico.

## Step 5: Caching di DataFrame

Un DataFrame riutilizzato più volte (es. in analisi interattive) può essere tenuto in memoria in formato colonnare compresso.
Eseguiamo due query sullo stesso DataFrame arricchito, prima senza cache e poi dopo `cache().count()` (che materializza la cache); alla fine `unpersist()` libera la memoria.

In [ ]:
arricchiti = ordini.join(prodotti, "prodotto_id").withColumn("importo", F.col("quantita") * F.col("prezzo"))

cronometra("senza cache - query 1", lambda: arricchiti.groupBy("categoria").sum("importo").collect())
cronometra("senza cache - query 2", lambda: arricchiti.groupBy("canale").sum("importo").collect())
arricchiti.cache().count()
cronometra("con cache   - query 1", lambda: arricchiti.groupBy("categoria").sum("importo").collect())
cronometra("con cache   - query 2", lambda: arricchiti.groupBy("canale").sum("importo").collect())
arricchiti.unpersist()

### Punti chiave
* `cache()` è lazy: serve un'azione (`count()`) per riempire la cache.
* Conviene solo per DataFrame riusati più volte; ricordarsi di `unpersist()`.

## Step 6: Chiudere la SparkSession

`spark.stop()` rilascia le risorse.

In [ ]:
spark.stop()

## Riepilogo

| Concetto | API / comando | A cosa serve |
|---|---|---|
| Partizioni di shuffle | `spark.sql.shuffle.partitions` | numero di partizioni dopo `groupBy`/`join` |
| AQE | `spark.sql.adaptive.enabled` | unire a runtime le partizioni troppo piccole |
| Join | `F.broadcast()`, `hint("merge")`, `autoBroadcastJoinThreshold` | scegliere tra broadcast hash e sort-merge |
| Ripartizionamento | `repartition(n)`, `coalesce(n)` | bilanciare o ridurre le partizioni |
| Caching | `cache()`, `unpersist()` | riusare un DataFrame in memoria |
| Chiusura | `spark.stop()` | rilasciare le risorse |